# 01 — Dask Benchmark

This notebook replicates the **Dask** side of the experiment described in
[*Benchmark: Koalas (PySpark) and Dask*](https://www.databricks.com/blog/2021/04/27/benchmark-koalas-pyspark-and-dask.html)
by Xinrong Meng and Hyukjin Kwon.

We run all **15 operations** under three scenarios:

| Scenario | Code |
|---|---|
| **Standard** | `operations(df)` |
| **Filtered** | `operations(df[(df.tip_amt >= 1) & (df.tip_amt < 5)])` |
| **Filtered + Cached** | cache filtered df → `operations(df_cached)` |

Results are saved to `results/dask_benchmark.csv`.

## 1  Setup

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import dask.dataframe as dd
from dask.distributed import Client, wait

from config import parquet_path, RESULTS_DIR, TIP_FILTER_MIN, TIP_FILTER_MAX
from src.benchmark_utils import BenchmarkTimer, build_results_table, print_table
from src.operations_dask import run_operations, read_parquet_timing

os.makedirs(RESULTS_DIR, exist_ok=True)
print('dask    :', dd.__version__)
print('pandas  :', pd.__version__)
print('numpy   :', np.__version__)

### 1a  Start Dask client

**Local execution** (single VM / laptop): use `LocalCluster` with all available cores.  
**Distributed execution** (Dataproc / multiple VMs): replace with the scheduler address.

In [ ]:
SCHEDULER_ADDRESS = os.environ.get('DASK_SCHEDULER', None)

if SCHEDULER_ADDRESS:
    # Distributed mode
    client = Client(SCHEDULER_ADDRESS)
    print(f'Connected to remote scheduler: {SCHEDULER_ADDRESS}')
else:
    # Local mode: use all available cores
    import multiprocessing
    n_workers = multiprocessing.cpu_count()
    client = Client(n_workers=n_workers, threads_per_worker=1, memory_limit='auto')
    print(f'Local scheduler with {n_workers} workers')

print(client)

## 2  Load data

In [ ]:
DATA_PATH = parquet_path('yellow_taxi.parquet')
print(f'Reading: {DATA_PATH}')

# Time the initial read as one of the 15 benchmark operations
t_read = read_parquet_timing(DATA_PATH, label='dask_standard')

# Load for the rest of the benchmark
df = dd.read_parquet(DATA_PATH)
print(f'Partitions: {df.npartitions}')
print(f'Columns   : {list(df.columns)}')
print(f'Rows      : {len(df):,}')

## 3  Scenario 1 — Standard operations

```python
operations(df)   # no filtering, no caching
```

In [ ]:
print('=== SCENARIO 1: STANDARD OPERATIONS ===')
t_std = run_operations(df, label='dask_standard')
# Merge the read_parquet timing from above
t_std._results['read_parquet'] = t_read._results.get('read_parquet', float('nan'))
print_table(t_std.to_series().to_frame(), title='Dask — Standard operations')

## 4  Scenario 2 — Operations with filtering

```python
# Filter is lazily composed with each operation
df_filtered = df[(df.tip_amt >= 1) & (df.tip_amt < 5)]
operations(df_filtered)
```

The filter keeps ~36 % of rows (tips between \$1 and \$5).

In [ ]:
df_filtered = df[(df['tip_amt'] >= TIP_FILTER_MIN) & (df['tip_amt'] < TIP_FILTER_MAX)]

print('=== SCENARIO 2: OPERATIONS WITH FILTERING ===')
t_flt = run_operations(df_filtered, label='dask_filtered')
print_table(t_flt.to_series().to_frame(), title='Dask — Filtered operations')

## 5  Scenario 3 — Operations with filtering and caching

```python
# Dask caching: persist filtered df in distributed memory
df_cached = client.persist(df_filtered)
wait(df_cached)            # block until fully materialised
operations(df_cached)
```

In [ ]:
print('Caching filtered DataFrame …')
df_cached = client.persist(df_filtered)
wait(df_cached)   # blocks until workers have materialised all partitions
print('Cache ready.')

print('=== SCENARIO 3: OPERATIONS WITH FILTERING + CACHING ===')
t_cache = run_operations(df_cached, label='dask_cached')
print_table(t_cache.to_series().to_frame(), title='Dask — Filtered+Cached operations')

## 6  Results summary

In [ ]:
import matplotlib.pyplot as plt

results_df = build_results_table(t_std, t_flt, t_cache)
print_table(results_df, title='DASK BENCHMARK — All scenarios (seconds)')

# Save
csv_path = os.path.join(RESULTS_DIR, 'dask_benchmark.csv')
results_df.to_csv(csv_path)
print(f'Saved: {csv_path}')

In [ ]:
# Bar chart comparison across scenarios
ax = results_df.plot(
    kind='bar', figsize=(14, 6),
    title='Dask — Elapsed time per operation (seconds)',
    ylabel='Time (s)', rot=45
)
ax.legend(['Standard', 'Filtered', 'Filtered+Cached'])
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'dask_benchmark.png'), dpi=120)
plt.show()

## 7  Notes on Dask execution

- **Lazy evaluation**: Dask builds a task graph for each expression.  Execution is triggered only when `.compute()` is called (or implicitly by scalar operations like `len()`).
- **Caching**: `client.persist()` materialises partitions into worker memory.  Subsequent operations skip re-reading from disk.
- **Partitioning**: each Parquet file becomes a Dask partition.  More partitions → better parallelism but higher scheduling overhead.
- **Scheduler**: the `distributed.Client` provides a full task scheduler with a web dashboard (usually at `http://localhost:8787`).

In [ ]:
client.close()